# sme testing : example 1 

resources: https://www.sympy.org/scipy-2017-codegen-tutorial/notebooks/07-the-hard-way.html

To test this API we start off with a simple redudant ODE

In [6]:
import sympy as sym

# Define state variables
x, y = sym.symbols('x y')

# Create a heavily redundant RHS vector (system of ODE's we can simplfiy) 
f1 = sym.sin(x + y) * sym.cos(x + y) + sym.exp(x + y)
f2 = sym.sin(x + y)**2 + sym.cos(x + y)**2 + sym.exp(x + y)

rhs = sym.Matrix([f1, f2])

print("rhs is:", rhs)

rhs is: Matrix([[exp(x + y) + sin(x + y)*cos(x + y)], [exp(x + y) + sin(x + y)**2 + cos(x + y)**2]])


In [16]:
print("original equation:", rhs)

# Execute Common Subexpression Elimination
replacements, reduced_rhs = sym.cse(rhs)

print("displaying replacements, temporary memory which can be utilised across the computation" 
"instead of computing (x+y) for instance again we compute it once and store it temporarily")
for var, expr in replacements:
    print(f"{var} = {expr}")

print("\n optimised and reduced RHS version:", (reduced_rhs[0]))

original equation: Matrix([[exp(x + y) + sin(x + y)*cos(x + y)], [exp(x + y) + sin(x + y)**2 + cos(x + y)**2]])
displaying replacements, temporary memory which can be utilised across the computationinstead of computing (x+y) for instance again and again
x0 = x + y
x1 = exp(x0)
x2 = sin(x0)
x3 = cos(x0)

 optimised and reduced RHS version: Matrix([[x1 + x2*x3], [x1 + x2**2 + x3**2]])


# sme testing : example 2

Second example we can look into non-linear equations that will need a jacobian to solve. Reminder of a jacobian? 

A Jacobian matrix is a grid of all the first-order partial derivatives of your equations. Essentially, it tells you how fast every equation (f1,f2) changes with respect to every state variable (x,y) Why do we need it for non-linear ODEs? Non-linear ODE systems are notoriously "stiff" or difficult to solve. Standard solvers fail or become incredibly slow. Advanced, stable solvers (like CVODE or Radau) rely on implicit mathematical methods (like Newton-Raphson) to take large, stable steps forward in time. Newton's method requires the Jacobian matrix at every single time-step to approximate the non-linear curves as local straight lines.

In [18]:
# utilising previously defined f1 and f2 equations and matrix


# compute the Jacobian with respect to state variables [x, y]
state_vars = sym.Matrix([x, y])
jacobian_matrix = rhs.jacobian(state_vars)

# apply CSE to the jacobian matrix
jac_replacements, jac_optimized = sym.cse(jacobian_matrix)

# Print results
print(" jacobian replacements ")
for var, expr in jac_replacements:
    print(f"{var} = {expr}")

print("\n optimised jacobian ")
print(jac_optimized[0])

 jacobian replacements 
x0 = x + y
x1 = exp(x0)
x2 = x1 - sin(x0)**2 + cos(x0)**2

 optimised jacobian 
Matrix([[x2, x2], [x1, x1]])


# ODE toolbox implementation 

If we implement this into the ODE toolbox, what do we need to think about? 

Symbolic Model Layer: Users define states, parameters, and algebraic equations.Compiler Engine: Takes the symbolic model, derives the analytical Jacobian, applies CSE with custom dummy prefixes, and exports code strings.Numerical Execution Layer: Loads the compiled strings, wraps them into a callable structure, and pipes them into a fast engine like SciPy's solve_ivp or a native C solver like Sundials CVODE.

Automated Code Generation (The Bridging Layer)You cannot pass a SymPy Matrix or CSE list directly to a numeric solver. You must translate the optimized equations into compiled code strings.For Python Backends: Use sym.lambdify with the cse=True flag, which handles all replacements internally and outputs optimized NumPy arrays.For C/C++ Backends: Use SymPy's ccode printer to turn the replacements list into standard C array mutations (double x0 = ...;).
